# Calculo del Índice de Fisher para la Energía


In [40]:
import openpyxl as px
import pandas as pd
import numpy as np

In [47]:
data = pd.read_excel('cantidad_depurada.xlsx')
data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   TIME              41 non-null     int64  
 1   PETRO_IND         41 non-null     float64
 2   GAS_IND           41 non-null     float64
 3   ELEC_IND          41 non-null     float64
 4   PETRO_HOG         41 non-null     float64
 5   GAS_HOG           41 non-null     float64
 6   ELEC_HOG          41 non-null     float64
 7   ELEC_HOG_PRECIO   41 non-null     float64
 8   GAS_HOG_PRECIO    41 non-null     float64
 9   PETRO_HOG_PRECIO  41 non-null     float64
 10  ELEC_IND_PRECIO   41 non-null     float64
 11  GAS_IND_PRECIO    41 non-null     float64
 12  PETRO_IND_PRECIO  41 non-null     float64
dtypes: float64(12), int64(1)
memory usage: 4.3 KB


In [48]:
# %% Fix TIME (años int64) y fijar base 2015
import pandas as pd

BASE_YEAR = 2015

data = data.copy()

# 1) Asegura que TIME son años numéricos válidos
time_year = pd.to_numeric(data['TIME'], errors='coerce')
if time_year.isna().any():
    bad_rows = data.loc[time_year.isna(), 'TIME'].head(5).tolist()
    raise ValueError(f"TIME contiene valores no numéricos (ejemplos: {bad_rows})")

# 2) Convierte a datetime (1 de enero del año)
data['TIME'] = pd.to_datetime(time_year.astype(int).astype(str) + '-01-01', format='%Y-%m-%d', errors='raise')

# 3) Ordena por tiempo
data = data.sort_values('TIME').reset_index(drop=True)

# 4) Selecciona base = primer periodo del año BASE_YEAR disponible
mask_base = data['TIME'].dt.year.eq(BASE_YEAR)
if not mask_base.any():
    years_present = sorted(data['TIME'].dt.year.unique().tolist())
    raise ValueError(f"No hay observaciones en {BASE_YEAR}. Años disponibles: {years_present}")

base_timestamp = data.loc[mask_base, 'TIME'].min()  # p.ej. 2015-01-01
print(f"Periodo base fijado en {base_timestamp} (BASE_YEAR={BASE_YEAR})")




Periodo base fijado en 2015-01-01 00:00:00 (BASE_YEAR=2015)


In [49]:
# %% Panel largo (t,i,j,P,Q) a partir de `data`
import pandas as pd
import numpy as np

# Mapeos de columnas (cantidades y precios)
Q_cols = {
    ('PETRO','IND'): 'PETRO_IND',
    ('GAS','IND'):   'GAS_IND',
    ('ELEC','IND'):  'ELEC_IND',
    ('PETRO','HOG'): 'PETRO_HOG',
    ('GAS','HOG'):   'GAS_HOG',
    ('ELEC','HOG'):  'ELEC_HOG',
}
P_cols = {
    ('PETRO','IND'): 'PETRO_IND_PRECIO',
    ('GAS','IND'):   'GAS_IND_PRECIO',
    ('ELEC','IND'):  'ELEC_IND_PRECIO',
    ('PETRO','HOG'): 'PETRO_HOG_PRECIO',
    ('GAS','HOG'):   'GAS_HOG_PRECIO',
    ('ELEC','HOG'):  'ELEC_HOG_PRECIO',
}

# Construcción en largo
rows_Q, rows_P = [], []
for (i,j), col in Q_cols.items():
    tmp = data[['TIME', col]].rename(columns={col:'Q'})
    tmp['i'], tmp['j'] = i, j
    rows_Q.append(tmp)

for (i,j), col in P_cols.items():
    tmp = data[['TIME', col]].rename(columns={col:'P'})
    tmp['i'], tmp['j'] = i, j
    rows_P.append(tmp)

long_Q = pd.concat(rows_Q, ignore_index=True)
long_P = pd.concat(rows_P, ignore_index=True)

panel = (long_Q.merge(long_P, on=['TIME','i','j'], how='outer')
                .rename(columns={'TIME':'t'})
                .sort_values(['t','i','j'])
                .reset_index(drop=True))

# Tipos y limpieza básica
panel['P'] = pd.to_numeric(panel['P'], errors='coerce')
panel['Q'] = pd.to_numeric(panel['Q'], errors='coerce')

# Para seguridad: elimina filas sin ningún precio (no aportan a índices)
panel_nonullP = panel.dropna(subset=['P']).copy()

panel_nonullP.head(12)




,t,Q,i,j,P
0,1978-01-01,2.012812,ELEC,HOG,306.976700
1,1978-01-01,4.164476,ELEC,IND,150.010000
2,1978-01-01,0.212051,GAS,HOG,115.122300
3,1978-01-01,1.132538,GAS,IND,66.111200
4,1978-01-01,1.339745,PETRO,HOG,70.428567
5,1978-01-01,2.130824,PETRO,IND,60.297368
6,1979-01-01,2.023021,ELEC,HOG,306.976700
7,1979-01-01,4.185598,ELEC,IND,150.010000
8,1979-01-01,0.212051,GAS,HOG,152.955700
9,1979-01-01,1.132538,GAS,IND,68.944500


In [50]:
# %% compute_chain_indices con encadenado hacia delante y hacia atrás desde base_t
import numpy as np
import pandas as pd

def compute_chain_indices(df_long, base_t, impute='ffill'):
    """
    Calcula:
      - Q_ratio: razones Fisher paso a paso (tm1->t), NaN en base_t
      - Q_t: índice de cantidad encadenado (base_t = 1)
      - nominal = ∑ P_t Q_t
      - E_t = Q_t * (∑ P_base Q_base)
      - P_t = nominal / E_t
    Encadena hacia delante y hacia atrás respecto a base_t.
    """
    df = df_long.copy()
    ts = sorted(df['t'].unique())
    if base_t not in ts:
        raise ValueError("El periodo base no existe en df_long.")

    # Gasto nominal por periodo
    nom_by_t = (df.assign(v=lambda d: d['P']*d['Q'])
                  .groupby('t')['v'].sum()
                  .reindex(ts))

    # Razones Fisher entre periodos adyacentes (tm1 -> t)
    # Las calculamos on-demand cuando se necesiten
    def step_ratio(t, tm1):
        return fisher_ratio_one_step(df, t, tm1, impute=impute)

    # Índice de cantidad encadenado
    Q_level = {base_t: 1.0}
    Q_ratio = {base_t: np.nan}

    # Encadenar hacia delante: base_idx+1 ... fin
    b_idx = ts.index(base_t)
    for k in range(b_idx + 1, len(ts)):
        t, tm1 = ts[k], ts[k - 1]
        r = step_ratio(t, tm1)
        Q_ratio[t] = r
        Q_level[t] = Q_level[tm1] * r if (r is not None and not np.isnan(r)) else np.nan

    # Encadenar hacia atrás: base_idx-1 ... 0
    for k in range(b_idx - 1, -1, -1):
        t, tp1 = ts[k], ts[k + 1]  # queremos r_{tp1}/_{t}
        r_forward = step_ratio(tp1, t)  # ratio (t -> tp1)
        Q_ratio[tp1] = Q_ratio.get(tp1, r_forward)  # ya estaba puesto en loop forward normalmente
        Q_level[t] = Q_level[tp1] / r_forward if (r_forward is not None and not np.isnan(r_forward)) else np.nan

    base_nominal = nom_by_t.loc[base_t]
    E_t = {t: (Q_level[t] * base_nominal) if Q_level[t] is not None else np.nan for t in ts}
    P_defl = {t: (nom_by_t.loc[t] / E_t[t]) if (E_t[t] is not None and E_t[t] != 0) else np.nan for t in ts}

    out = pd.DataFrame({
        't': ts,
        'Q_ratio': [Q_ratio.get(t, np.nan) for t in ts],
        'Q_t': [Q_level.get(t, np.nan) for t in ts],
        'nominal': [nom_by_t.loc[t] for t in ts],
        'E_t': [E_t[t] for t in ts],
        'P_t': [P_defl[t] for t in ts],
    })
    return out


In [51]:
# %% Índices agregados (toda energía × todos los usos), base 100 en 2015
# Usa el panel con P no nulo
panel_full = panel_nonullP.copy()

out_total = compute_chain_indices(panel_full, base_t=base_timestamp, impute='ffill')

# Reescalados a base 100 en el periodo base
out_total['Q_t_100']   = (out_total['Q_t'] / out_total.loc[out_total['t'].eq(base_timestamp), 'Q_t'].iloc[0]) * 100
out_total['P_idx_100'] = (out_total['P_t'] / out_total.loc[out_total['t'].eq(base_timestamp), 'P_t'].iloc[0]) * 100
out_total['E_idx_100'] = (out_total['E_t'] / out_total.loc[out_total['t'].eq(base_timestamp), 'E_t'].iloc[0]) * 100

out_total.head(12)




,t,Q_ratio,Q_t,nominal,E_t,P_t,Q_t_100,P_idx_100,E_idx_100
0,1978-01-01,NaN,0.443986,1564.724091,16262.670715,0.096216,44.398610,9.621569,44.398610
1,1979-01-01,1.003964,0.445746,1622.309478,16327.132935,0.099363,44.574598,9.936279,44.574598
2,1980-01-01,1.002750,0.446972,2393.905920,16372.036592,0.146219,44.697189,14.621919,44.697189
3,1981-01-01,0.992710,0.443713,3500.267591,16252.682356,0.215366,44.371341,21.536553,44.371341
4,1982-01-01,1.050306,0.466035,4189.306859,17070.293569,0.245415,46.603495,24.541504,46.603495
5,1983-01-01,1.067937,0.497696,5502.906257,18230.000917,0.301860,49.769605,30.185990,49.769605
6,1984-01-01,1.022675,0.508981,6311.508292,18643.366359,0.338539,50.898131,33.853909,50.898131
7,1985-01-01,1.004240,0.511139,6784.255669,18722.407829,0.362360,51.113922,36.236021,51.113922
8,1986-01-01,1.034981,0.529019,7083.005143,19377.332777,0.365530,52.901928,36.553045,52.901928
9,1987-01-01,1.135309,0.600601,7961.777475,21999.267404,0.361911,60.060054,36.191103,60.060054


In [52]:
# %% Subíndices por uso (HOG / IND), base 100 en 2015
subs_use = {}
for jj in sorted(panel_full['j'].dropna().unique()):
    df_j = panel_full[panel_full['j'].eq(jj)]
    out_j = compute_chain_indices(df_j, base_t=base_timestamp, impute='ffill')
    out_j['Q_t_100']   = (out_j['Q_t'] / out_j.loc[out_j['t'].eq(base_timestamp), 'Q_t'].iloc[0]) * 100
    out_j['P_idx_100'] = (out_j['P_t'] / out_j.loc[out_j['t'].eq(base_timestamp), 'P_t'].iloc[0]) * 100
    out_j['E_idx_100'] = (out_j['E_t'] / out_j.loc[out_j['t'].eq(base_timestamp), 'E_t'].iloc[0]) * 100
    subs_use[jj] = out_j

# Ejemplos
subs_use['HOG'].head(10), subs_use['IND'].head(10)


(           t   Q_ratio       Q_t      nominal           E_t       P_t  \
 0 1978-01-01       NaN  0.332166   736.654559   8011.751292  0.091947   
 1 1979-01-01  1.004184  0.333556   764.935160   8045.271813  0.095079   
 2 1980-01-01  1.003023  0.334564  1066.147299   8069.589189  0.132119   
 3 1981-01-01  0.989139  0.330930  1506.882813   7981.949113  0.188786   
 4 1982-01-01  1.036982  0.343169  1742.341049   8277.139105  0.210500   
 5 1983-01-01  1.050439  0.360478  2368.379605   8694.631051  0.272396   
 6 1984-01-01  1.018276  0.367066  2717.726260   8853.537142  0.306965   
 7 1985-01-01  1.012358  0.371603  3006.492880   8962.949875  0.335436   
 8 1986-01-01  1.032202  0.383569  3227.348007   9251.578238  0.348843   
 9 1987-01-01  1.136694  0.436000  3599.223204  10516.209313  0.342255   
 
      Q_t_100  P_idx_100  E_idx_100  
 0  33.216597   9.194676  33.216597  
 1  33.355572   9.507885  33.355572  
 2  33.456392  13.211915  33.456392  
 3  33.093038  18.878632  33.093

In [53]:
# %% Subíndices por energía (PETRO / GAS / ELEC), base 100 en 2015
subs_energy = {}
for ii in sorted(panel_full['i'].dropna().unique()):
    df_i = panel_full[panel_full['i'].eq(ii)]
    out_i = compute_chain_indices(df_i, base_t=base_timestamp, impute='ffill')
    out_i['Q_t_100']   = (out_i['Q_t'] / out_i.loc[out_i['t'].eq(base_timestamp), 'Q_t'].iloc[0]) * 100
    out_i['P_idx_100'] = (out_i['P_t'] / out_i.loc[out_i['t'].eq(base_timestamp), 'P_t'].iloc[0]) * 100
    out_i['E_idx_100'] = (out_i['E_t'] / out_i.loc[out_i['t'].eq(base_timestamp), 'E_t'].iloc[0]) * 100
    subs_energy[ii] = out_i

# Ejemplos
subs_energy['PETRO'].head(10), subs_energy['GAS'].head(10), subs_energy['ELEC'].head(10)



(           t   Q_ratio       Q_t      nominal          E_t       P_t  \
 0 1978-01-01       NaN  0.596318   222.839376  2689.894079  0.082843   
 1 1979-01-01  1.000000  0.596318   262.891069  2689.894079  0.097733   
 2 1980-01-01  0.994176  0.592845   472.172382  2674.227835  0.176564   
 3 1981-01-01  0.871119  0.516439   638.608664  2329.570469  0.274132   
 4 1982-01-01  1.185609  0.612294   602.425227  2761.958801  0.218115   
 5 1983-01-01  1.271129  0.778304  1276.540201  3510.805259  0.363603   
 6 1984-01-01  1.014279  0.789418  1429.457447  3560.937240  0.401427   
 7 1985-01-01  1.048834  0.827969  1631.948587  3734.832547  0.436954   
 8 1986-01-01  1.072987  0.888399  1487.469438  4007.425191  0.371178   
 9 1987-01-01  1.538311  1.366635  1943.070140  6164.666977  0.315195   
 
       Q_t_100  P_idx_100   E_idx_100  
 0   59.631807   8.284318   59.631807  
 1   59.631807   9.773287   59.631807  
 2   59.284504  17.656401   59.284504  
 3   51.643853  27.413151   51.6438

In [56]:
# %% Exportación CSV adaptada a configuración española
import os

os.makedirs('resultados_indices', exist_ok=True)

def exportar_csv_es(df, nombre):
    """
    Exporta DataFrame en formato CSV con:
      - delimitador ';'
      - separador decimal ','
      - codificación 'utf-8-sig'
    """
    ruta = f"resultados_indices/{nombre}.csv"
    df.to_csv(
        ruta,
        sep=';',              # separador de columnas
        decimal=',',          # separador decimal
        encoding='utf-8-sig', # compatible con Excel
        index=False
    )
    print(f"✅ Exportado: {ruta}")

# === 1) Total agregado ===
exportar_csv_es(out_total, "indices_total_fisher_base2015")

# === 2) Subíndices por uso ===
for jj, df in subs_use.items():
    exportar_csv_es(df, f"indices_use_{jj}_fisher_base2015")

# === 3) Subíndices por energía ===
for ii, df in subs_energy.items():
    exportar_csv_es(df, f"indices_energy_{ii}_fisher_base2015")




✅ Exportado: resultados_indices/indices_total_fisher_base2015.csv
✅ Exportado: resultados_indices/indices_use_HOG_fisher_base2015.csv
✅ Exportado: resultados_indices/indices_use_IND_fisher_base2015.csv
✅ Exportado: resultados_indices/indices_energy_ELEC_fisher_base2015.csv
✅ Exportado: resultados_indices/indices_energy_GAS_fisher_base2015.csv
✅ Exportado: resultados_indices/indices_energy_PETRO_fisher_base2015.csv
